In [ ]:
from pathlib import Path
from datetime import datetime, date
import xarray as xr

from wekeo_combined_chain import config

def get_combined_ds(date: date) -> xr.Dataset:
    """
    TODO: replace with S3 access instead
    """
    base = config.gridded_combined_downloaded_dir
    file = base / date.strftime("%Y_%m_%d/COMBINED_%Y-%m-%d_v1.0.nc")
    return xr.open_dataset(file)


day = date(2025, 8, 2)

ds = get_combined_ds(day)

print(ds)

In [ ]:
areas = {
    "Global":          [ 90., -90.,  180., -180.],
    "North_America":   [ 90.,   9.,  -20., -169.],
    "South_America":   [  9., -60.,  -20.,  120.],
    "Europe":          [ 90.,  36.,   31.,  -20.],
    "Africa":          [ 36., -60.,   60.,  -20.],
    "Russia":          [ 90.,  36., -169.,   31.],
    "Asia":            [ 36., -10., -169.,   60.],
    "Australia":       [-10., -60., -120.,   60.],
    # "Central_Pacific": [  9., -10., -120., -169.],
    # "Antarctic":       [-60., -90.,  180., -180.],
}
    
from wekeo_combined_chain.utils import select_area

area_name = "Global"
area = areas[area_name]
    
ds_area = select_area(ds, area)

In [ ]:
from wekeo_combined_chain import postprocess

ds_post, df_plumes = postprocess.compute(ds_area)

In [ ]:
from wekeo_combined_chain.postprocess import table as T

#print("Plumes:")
print("-" * 75)
#print(T.plume_table(df_plumes))
# MAJ 11/06/26
T.display_plume_tables(df_plumes)

print()
#print("Tiny plumes:")
print("-" * 75)
# MAJ 11/06/26
#print(T.tiny_plume_table(df_plumes))
T.display_tiny_plume_tables(df_plumes)

In [ ]:
from wekeo_combined_chain.postprocess import plot as P

date_str = day.strftime("%Y%m%d")
output_dir = Path("output") / day.strftime("%Y_%m_%d") / area_name
output_dir.mkdir(parents=True, exist_ok=True)


## Map 1 — Plumes × FRP overlay

In [ ]:
# Toggle save_to to write to disk, or leave None for inline display only
P.plot_plumes_frp(ds_area, ds_post, date_str, frp_channel="SWIR",
                  save_to=output_dir / f"plumes_frp_SWIR_{date_str}.png")


## Map 2 — Fire score per plume

In [ ]:
P.plot_fire_score_plume(ds_area, ds_post, df_plumes, date_str, band="MWIR",
                        save_to=output_dir / f"fire_score_plume_MWIR_{date_str}.png")


## Map 3 — Fire score per pixel

In [ ]:
P.plot_fire_score_pixel(ds_area, ds_post, date_str, band="MWIR",
                        save_to=output_dir / f"fire_score_pixel_SWIR_{date_str}.png")
    

## Map 4 — Plume envelopes + source confidence

In [ ]:
P.plot_plume_envelopes(ds_area, ds_post, df_plumes, date_str, frp_channel="MWIR",
                       save_to=output_dir / f"plume_envelopes_MWIR_{date_str}.png")


In [ ]:
# end of demo